In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import difflib
import re

In [6]:
runners = pd.read_excel("../../data/cleaned/results/runners.xlsx")

In [16]:
highest_ratings = pd.read_excel("../../data/cleaned/acceptances/highest_ratings.xlsx")

In [ ]:
# 1. Get unique names and clean them to find potential duplicates
unique_names = highest_ratings.horse_name.dropna().unique()

# 2. Look for close matches
similar_pairs = []
for i, name1 in enumerate(unique_names):
    # Get close matches from the rest of the list
    matches = difflib.get_close_matches(name1, unique_names[i+1:], n=3, cutoff=0.8)
    for match in matches:
        similar_pairs.append((name1, match))

# 3. View the results clearly
df_typos = pd.DataFrame(similar_pairs, columns=['Name A', 'Name B'])
df_typos

,Name A,Name B
0,SPARKLING THEA,SPARKLING DEW
1,RARE SILVER,STAR SILVER
2,ADELE,ADELINE
3,LUCAS,LUCA
4,GOLDEN TOWER,GOLDEN ERA
5,DIVINE SPARK,DIVINE STAR
6,GOLDEN EVE,GOLDEN ERA
7,ZANARA,ANAIRA
8,MARIANELLA,MARIELLA
9,WESTERN STYLE,WESTERN STAR


In [8]:
unique_names = highest_ratings = pd.read_excel("../../data/cleaned/acceptances/highest_ratings.xlsx").horse_name.dropna().unique()

# Create a DataFrame of unique names
df_names = pd.DataFrame({'original_name': unique_names})

# Create a 'clean' key: uppercase, no spaces, no punctuation
df_names['clean_key'] = df_names['original_name'].apply(
    lambda x: re.sub(r'[^A-Z0-9]', '', str(x).upper())
)

# Find clean keys that appear more than once (meaning they have variations)
duplicates = df_names[df_names.duplicated(subset=['clean_key'], keep=False)]

# Sort them so variations sit next to each other
duplicates.sort_values(by='clean_key')

,original_name,clean_key


In [22]:
# =========================
# 2. NORMALIZE
# =========================
def normalize(df):
    df = df.copy()
    df['meet_date'] = pd.to_datetime(df['meet_date'])
    df['horse_name'] = df['horse_name'].str.strip().str.upper()
    return df

runners = normalize(runners)

# Load and normalize raw highest_ratings directly to avoid index errors
raw_highest_ratings = pd.read_excel("../../data/cleaned/acceptances/highest_ratings.xlsx")
highest_ratings = normalize(raw_highest_ratings)

# =========================
# 3. DETECT DUPLICATES IN RUNNERS (RISK)
# =========================
dup_check = runners.groupby(['meet_date', 'horse_name']).size().reset_index(name='count')
duplicate_horses = dup_check[dup_check['count'] > 1]

print("Potential ambiguous horses in runners:", len(duplicate_horses))
display(duplicate_horses.head())

# =========================
# 4. BUILD MAPPING (GROUND TRUTH)
# =========================
runner_map = runners[
    ['meet_date', 'horse_name', 'race_no']
].drop_duplicates()

# =========================
# 5. MERGE (Keeps all highest_ratings)
# =========================
merged = highest_ratings.merge(
    runner_map,
    on=['meet_date', 'horse_name'],
    how='left',
    indicator=True,
    suffixes=('_old', '_true')
)

# =========================
# 6. VALIDATE AND FIX RACE_NO
# =========================
# Create a boolean column tracking if the horse exists in the runners file
merged['is_valid_runner'] = merged['_merge'] == 'both'

# Update race_no only for found runners; keep old race_no for non-runners
merged['race_no'] = merged['race_no_true'].fillna(merged['race_no_old'])

# Clean up temporary merge columns
clean_highest_ratings = merged.drop(
    columns=['_merge', 'race_no_old', 'race_no_true']
)

# =========================
# 7. GENERATE LISTS FOR INSPECTION
# =========================
# List 1: Valid Runners
valid_runners_list = clean_highest_ratings[clean_highest_ratings['is_valid_runner'] == True][['meet_date', 'horse_name']].drop_duplicates()
print(f"Valid Runners Found: {len(valid_runners_list)}")
display(valid_runners_list.head())

# List 2: Non-Runners (Kept in the main file instead of dropped)
non_runners_list = clean_highest_ratings[clean_highest_ratings['is_valid_runner'] == False][['meet_date', 'horse_name']].drop_duplicates()
print(f"Kept Non-Runners (Flagged as Invalid): {len(non_runners_list)}")
display(non_runners_list.head())

# =========================
# 8. FINAL SORT AND DATE FORMAT
# =========================
clean_highest_ratings = clean_highest_ratings.sort_values(
    by=['meet_date', 'race_no']
).reset_index(drop=True)

# Convert to string date format right before saving
clean_highest_ratings['meet_date'] = clean_highest_ratings['meet_date'].dt.strftime('%Y-%m-%d')
clean_highest_ratings['achieved_date'] = clean_highest_ratings['achieved_date'].dt.strftime('%Y-%m-%d')

# =========================
# 9. SAVE
# =========================
clean_highest_ratings.to_excel("../../data/cleaned/acceptances_cleaned/highest_ratings.xlsx", index=False)

Potential ambiguous horses in runners: 0


,meet_date,horse_name,count


Valid Runners Found: 21702


,meet_date,horse_name
0,2018-01-04,REMEMBER ME
1,2018-01-04,UNCLE SCROOGE
2,2018-01-04,GLORIOUS EYES
3,2018-01-04,DAFFODIL
4,2018-01-04,BOLD MARCH


Kept Non-Runners (Flagged as Invalid): 530


,meet_date,horse_name
2332,2018-08-25,BONAFIDE
2333,2018-08-25,BOURBON PRINCESS
2338,2018-08-25,JETFIRE
2339,2018-08-25,JOPLIN
2340,2018-08-25,HIGHLAND EMPRESS
